In [3]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.fftpack import fft, ifft, fftfreq
import pydicom
from pydicom.dataset import Dataset, FileDataset
from pydicom.uid import UID, generate_uid
import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
import time
import warnings

# Funkcja do ładowania obrazu (obsługuje DICOM i zwykłe obrazy)
def load_image(file_path):
    if file_path.lower().endswith('.dcm'):
        ds = pydicom.dcmread(file_path)
        image = ds.pixel_array.astype(np.float32)
        
        # Reskalowanie jeśli potrzebne
        if 'RescaleSlope' in ds:
            image *= ds.RescaleSlope
        if 'RescaleIntercept' in ds:
            image += ds.RescaleIntercept
            
        return image
    else:
        image = Image.open(file_path).convert('L')
        return np.array(image, dtype=np.float32)

# Funkcja do zapisu DICOM
def save_as_dicom(image_array, output_path, patient_info, study_date, comments):
    # Normalizacja i konwersja do uint16
    image_array = (image_array - np.min(image_array)) / (np.max(image_array) - np.min(image_array)) * 65535
    image_array = image_array.astype(np.uint16)

    # Podstawowe metadane
    file_meta = Dataset()
    file_meta.MediaStorageSOPClassUID = UID('1.2.840.10008.5.1.4.1.1.7')  # Secondary Capture
    file_meta.MediaStorageSOPInstanceUID = generate_uid()
    file_meta.ImplementationClassUID = generate_uid()
    file_meta.TransferSyntaxUID = UID('1.2.840.10008.1.2.1')  # Explicit VR Little Endian

    ds = FileDataset(output_path, {}, file_meta=file_meta, preamble=b"\0"*128)
    
    # Wymagane tagi DICOM
    ds.PatientName = patient_info.get('name', '')
    ds.PatientID = patient_info.get('id', '')
    ds.StudyDate = study_date if study_date else datetime.date.today().strftime('%Y%m%d')
    ds.StudyInstanceUID = generate_uid()
    ds.SeriesInstanceUID = generate_uid()
    ds.SOPInstanceUID = generate_uid()
    ds.Modality = "OT"
    
    # Sekwencja Study
    ds.StudyDescription = "Reconstructed CT"
    ds.add_new(0x00324000, 'LO', comments)  # Prawidłowy tag dla Study Comments
    
    # Informacje o serii
    ds.SeriesDate = datetime.date.today().strftime('%Y%m%d')
    ds.SeriesNumber = 1
    
    # Informacje o obrazie
    ds.Rows, ds.Columns = image_array.shape
    ds.SamplesPerPixel = 1
    ds.PhotometricInterpretation = "MONOCHROME2"
    ds.PixelRepresentation = 0
    ds.HighBit = 15
    ds.BitsStored = 16
    ds.BitsAllocated = 16
    ds.PixelData = image_array.tobytes()
    
    # Geometria
    ds.ImagePositionPatient = [0.0, 0.0, 0.0]  # Wartości numeryczne
    ds.ImageOrientationPatient = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0]
    ds.PixelSpacing = [1.0, 1.0]  # Wartości numeryczne
    ds.SliceThickness = 1.0
    
    # Dodatkowe wymagane tagi
    ds.ContentDate = datetime.date.today().strftime('%Y%m%d')
    ds.ContentTime = datetime.datetime.now().strftime('%H%M%S')
    ds.add_new(0x00080023, 'DA', ds.ContentDate)  # Content Date
    ds.add_new(0x00080033, 'TM', ds.ContentTime)  # Content Time
    
    ds.is_little_endian = True
    ds.is_implicit_VR = False
    
    # Zapis pliku
    ds.save_as(output_path)
    print(f"Zapisano plik DICOM: {output_path}")

# Funkcja transformacji Radona
def radon_transform(image, delta_alpha, n_detectors):
    h, w = image.shape
    diagonal = np.sqrt(h**2 + w**2)  # Przekątna obrazu
    spread = diagonal
    
    angles = np.deg2rad(np.arange(0, 180, delta_alpha))
    s = np.linspace(-spread/2, spread/2, n_detectors)
    sinogram = np.zeros((len(angles), n_detectors))
    
    center_x, center_y = w // 2, h // 2
    max_length = int(np.sqrt(h**2 + w**2))  # Długość przekątnej obrazu
    
    for i, theta in enumerate(angles):
        cos_t, sin_t = np.cos(theta), np.sin(theta)
        
        for j, s_val in enumerate(s):
            line_sum = 0.0
            for t in np.linspace(-max_length/2, max_length/2, max_length):
                x = int(np.round(center_x + s_val * cos_t + t * (-sin_t)))
                y = int(np.round(center_y + s_val * sin_t + t * cos_t))
                
                if 0 <= x < w and 0 <= y < h:
                    line_sum += image[y, x]
            
            sinogram[i, j] = line_sum
    
    return sinogram

# Filtr Ram-Lak
def apply_filter(sinogram):
    n = sinogram.shape[1]
    freqs = fftfreq(n).reshape(1, -1)
    filter_kernel = np.abs(freqs * n)  
    
    sinogram_fft = fft(sinogram, axis=1)
    filtered_sinogram = np.real(ifft(sinogram_fft * filter_kernel, axis=1))
    
    return filtered_sinogram

# Funkcja odwrotnej transformacji Radona
def inverse_radon_transform(sinogram, delta_alpha, n_detectors, original_h, original_w):
    h, w = original_h, original_w
    diagonal = np.sqrt(h**2 + w**2)
    spread = diagonal
    
    reconstructed = np.zeros((h, w))
    angles = np.deg2rad(np.arange(0, 180, delta_alpha))
    center_x = w // 2
    center_y = h // 2
    
    detector_spacing = spread / (n_detectors - 1) if n_detectors > 1 else 1
    s_center = (n_detectors - 1) / 2  # Środek w przestrzeni detektorów

    for i, theta in enumerate(angles):
        cos_t = np.cos(theta)
        sin_t = np.sin(theta)
        
        for x in range(w):
            for y in range(h):
                # Obliczenie współrzędnych w przestrzeni Radona
                s = (x - center_x) * cos_t + (y - center_y) * sin_t
                
                # Przeliczenie na indeks detektora z interpolacją
                s_idx = s / detector_spacing + s_center
                
                if 0 <= s_idx < n_detectors:
                    # Interpolacja liniowa
                    j = int(np.floor(s_idx))
                    a = s_idx - j
                    if j < n_detectors - 1:
                        reconstructed[y, x] += (1 - a) * sinogram[i, j] + a * sinogram[i, j+1]
                    else:
                        reconstructed[y, x] += sinogram[i, j]

    return reconstructed * np.pi / (2 * len(angles))

# Funkcja do obliczania MSE
def calculate_mse(original, reconstructed):
    # Normalizacja do zakresu [0, 1]
    original_norm = (original - np.min(original)) / (np.max(original) - np.min(original) + 1e-10)
    reconstructed_norm = (reconstructed - np.min(reconstructed)) / (np.max(reconstructed) - np.min(reconstructed) + 1e-10)
    
    return np.mean((original_norm - reconstructed_norm) ** 2)

# Funkcja do przetwarzania obrazu
def process_image(file_path, delta_alpha, n_detectors, dicom_output=None, patient_info=None, study_date=None, comments=None):
    image_array = load_image(file_path)
    h, w = image_array.shape
    
    # Przetwarzanie
    sinogram = radon_transform(image_array, delta_alpha, n_detectors)
    filtered_sinogram = apply_filter(sinogram)
    reconstructed_image = inverse_radon_transform(filtered_sinogram, delta_alpha, n_detectors, h, w)
    reconstructed_image_without_filter = inverse_radon_transform(sinogram, delta_alpha, n_detectors, h, w)

    # Zapis DICOM
    if dicom_output:
        if not patient_info:
            patient_info = {'name': 'Anonymous', 'id': '0'}
        if not study_date:
            study_date = datetime.date.today().strftime('%Y%m%d')
        if not comments:
            comments = "Reconstructed image from Radon transform"
            
        save_as_dicom(reconstructed_image, dicom_output, patient_info, study_date, comments)
        
    
    mse_without_filter = calculate_mse(image_array, reconstructed_image_without_filter)
    mse_with_filter = calculate_mse(image_array, reconstructed_image)
    
    print(f"\nAnaliza jakości:")
    print(f"MSE bez filtra: {mse_without_filter:.6f}")
    print(f"MSE z filtrem:  {mse_with_filter:.6f}")
    
    # Wizualizacja
    fig, axs = plt.subplots(2, 3, figsize=(16, 8))
    
    axs[0,0].imshow(image_array, cmap='gray')
    axs[0,0].set_title("Oryginalny obraz")

    axs[0,1].imshow(sinogram, cmap='gray', aspect='auto', extent=[0, 180, -np.sqrt(h**2 + w**2)/2, np.sqrt(h**2 + w**2)/2])
    axs[0,1].set_title("Sinogram")
    axs[0,1].set_xlabel("Kąt (stopnie)")
    axs[0,1].set_ylabel("Pozycja detektora")

    axs[0,2].imshow(filtered_sinogram, cmap='gray', aspect='auto', extent=[0, 180, -np.sqrt(h**2 + w**2)/2, np.sqrt(h**2 + w**2)/2])
    axs[0,2].set_title("Filtr Ram-Lak")
    axs[0,2].set_xlabel("Kąt (stopnie)")

    axs[1,0].imshow(reconstructed_image_without_filter, cmap='gray')
    axs[1,0].set_title("Bez filtru")

    axs[1,1].imshow(reconstructed_image, cmap='gray')
    axs[1,1].set_title("Rekonstrukcja Z filtrami")

    axs[1,2].imshow(image_array - reconstructed_image, cmap='gray')
    axs[1,2].set_title("Różnica między oryginałem a rekonstrukcją")

    plt.tight_layout()
    plt.show()
    
class CTProcessor:
    def __init__(self):
        # Widgety do wyboru parametrów
        self.file_selector = widgets.SelectMultiple(
            options=sorted(os.listdir('tomograf-dicom')),
            description='Wybierz obrazy:',
            rows=5,
            layout={'width': '400px'}
        )
        
        self.n_detectors = widgets.Dropdown(
            options=[100, 200, 300, 400, 500],
            value=200,
            description='Liczba detektorów:',
            style={'description_width': 'initial'}
        )
        
        self.process_btn = widgets.Button(
            description="Rozpocznij rekonstrukcję!",
            button_style='success',
            icon='cogs'
        )
        
        # Wskaźniki postępu
        self.progress_bar = widgets.IntProgress(
            value=0,
            min=0,
            max=100,
            description='Postęp całkowity:',
            bar_style='info',
            orientation='horizontal'
        )
        
        self.current_status = widgets.HTML(
            value="<i>Oczekiwanie na rozpoczęcie...</i>",
            placeholder='',
            description='Status:'
        )
        
        # Kontener wyjściowy
        self.output_area = widgets.Output()
        
        # Połączenie zdarzeń
        self.process_btn.on_click(self.start_processing)
        
    def create_gui(self):
        return widgets.VBox([
            widgets.HTML("<h2>🔬 Tomograf CT - Panel Sterowania</h2>"),
            widgets.HBox([
                self.file_selector,
                widgets.VBox([
                    self.n_detectors,
                    self.process_btn
                ])
            ]),
            self.progress_bar,
            self.current_status,
            self.output_area
        ])
    
    def update_status(self, message):
        self.current_status.value = f"<b>[{time.strftime('%H:%M:%S')}]</b> {message}"
    
    def start_processing(self, btn):
        selected_files = self.file_selector.value
        if not selected_files:
            with self.output_area:
                print("Proszę wybrać przynajmniej jeden plik DICOM!")
            return
        
        total_files = len(selected_files)
        self.progress_bar.max = total_files
        self.progress_bar.value = 0
        
        with self.output_area:
            clear_output()
            print("🔄 Rozpoczęcie procesu rekonstrukcji...\n")
            
            for idx, filename in enumerate(selected_files, 1):
                try:
                    # Aktualizacja statusu
                    self.progress_bar.value = idx
                    status_msg = f"Przetwarzanie pliku {idx}/{total_files}: {filename}"
                    self.update_status(status_msg)
                    
                    # Przygotowanie parametrów
                    input_path = os.path.join('tomograf-dicom', filename)
                    output_filename = f"RECON_{os.path.splitext(filename)[0]}.dcm"
                    output_path = os.path.join('wyniki', output_filename)
                    
                    params = {
                        'file_path': input_path,
                        'n_detectors': self.n_detectors.value,
                        'dicom_output': output_path,
                        'patient_info': {
                            'name': 'Kowalski^Jan',
                            'id': str(idx)
                        },
                        'delta_alpha': 1,
                        'study_date': '20250415',
                        'comments': f"Automatyczna rekonstrukcja: {filename}"
                    }
                    
                    # Wykonanie przetwarzania
                    start_time = time.time()
                    process_image(**params)
                    processing_time = time.time() - start_time
                    
                    print(f"✅ {filename}:")
                    print(f"   - Czas przetwarzania: {processing_time:.2f}s")
                    print(f"   - Detektory: {self.n_detectors.value}")
                    print(f"   - Wynik: {output_path}\n")
                    
                except Exception as e:
                    print(f"❌ Błąd w pliku {filename}:")
                    print(f"   - Typ błędu: {type(e).__name__}")
                    print(f"   - Komunikat: {str(e)}\n")
                    continue
                
            self.update_status("Przetwarzanie zakończone!")
            print("🏁 Wszystkie operacje wykonane!")

In [5]:
processor = CTProcessor()
display(processor.create_gui())